# S6E10 — 21 XGBoost with window target-rate features (fold-0 test)

> **Pipeline position:** 03 `X_final` → 05 XGBoost → **21 window target rates** → 11 stack

| | |
|---|---|
| **Input** | `data/processed/X_final_*.parquet`, `y_train.parquet`; OOF files of the stack members (mini-stack) |
| **Output** | `preds/xgb_window_folds/` and `preds/xgb_window_ref_folds/` (fold-0 checkpoints) |
| **Runtime** | ≈ 5 min on CPU |
| **Result** | see *Results & decisions* |

**Idea:** a *window target rate* is the satisfied share among passengers whose value lies **near** this passenger's value — e.g. all routes within ±50 km of this distance, or all ages within ±2 years. The exact target encoding of `flight_distance` (E12 in `02`) uses only the *same* value (≈ 200 rows); a window borrows strength from neighbouring values and sees a smooth trend a tree has to piece together from many splits. It is information about *other rows*, the only kind that has moved the score in this project. Counter-evidence: TE of coarse distance *bins* gave nothing for LightGBM (E15 in `02`). The window differs from a bin by being centred on each row.

**Leakage — the main risk.** If a training row's own label were inside its window rate, the tree would learn to trust the feature and fail on validation. Guards:
1. **In-fold:** statistics come only from the fold's training part; validation and test rows are only looked up.
2. **Cross-fitted for training rows:** the training part is split into 5 inner parts; each part's features are computed from the other 4 (same scheme as sklearn's `TargetEncoder(cv=5)`).
3. **Check:** each feature's AUC on its own is printed for training rows (cross-fitted) and for validation rows. A leak would show as a clearly higher AUC on the training rows.

**Paired comparison:** XGBoost with the parameters of `05`, on `X_final` + in-fold TE of the route, with and without the window features; same fold, same seed. Unlike `05`, early stopping uses 10% of the fold's *training* part, never the validation fold.

> **Glossary:** *Window target rate* = smoothed satisfied share over rows whose value is within ±w of this row's value. *Smoothing* = (sum + m·prior) / (count + m), pulls rates of thin windows towards the overall rate. *Cross-fitting* = a row's feature is computed by a model / statistic that never saw the row.

In [1]:
import numpy as np, pandas as pd, time, warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
warnings.filterwarnings('ignore')

PROC_DIR = next(p for p in [Path('data/processed'), Path('../data/processed')] if (p / 'X_final_train.parquet').exists())
PREDS_DIR = next(p for p in [Path('preds'), Path('../preds')] if p.exists())
X = pd.read_parquet(PROC_DIR / 'X_final_train.parquet'); X_test = pd.read_parquet(PROC_DIR / 'X_final_test.parquet')
y = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int).values
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class']
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y))
tr0, va0 = FOLDS[0]
# Windows: (column, half-width). Values are integers (km, years, minutes).
WINDOWS = [('flight_distance', 10), ('flight_distance', 50), ('flight_distance', 200), ('age', 2), ('age', 5), ('departure_delay_in_minutes', 5)]
M_SMOOTH = 50
print(X.shape, '| windows:', WINDOWS)

(699635, 45) | windows: [('flight_distance', 10), ('flight_distance', 50), ('flight_distance', 200), ('age', 2), ('age', 5), ('departure_delay_in_minutes', 5)]


## 1. Window target rates (in-fold, cross-fitted)
Counts per integer value via `bincount`, window sums via cumulative sums, so every window feature for a million rows takes milliseconds.

In [2]:
def window_stats(v_fit, y_fit, w, vmax):
    s = np.bincount(v_fit, weights=y_fit, minlength=vmax + 1); n = np.bincount(v_fit, minlength=vmax + 1).astype(float)
    cs, cn = np.r_[0, np.cumsum(s)], np.r_[0, np.cumsum(n)]
    lo = np.clip(np.arange(vmax + 1) - w, 0, vmax + 1); hi = np.clip(np.arange(vmax + 1) + w + 1, 0, vmax + 1)
    return cs[hi] - cs[lo], cn[hi] - cn[lo]                      # window sum of labels and window count, per value

def window_rate(v_fit, y_fit, v_apply, w, vmax):
    S, N = window_stats(v_fit, y_fit, w, vmax); prior = y_fit.mean()
    return ((S + M_SMOOTH * prior) / (N + M_SMOOTH))[v_apply].astype(np.float32)

def window_features(fit_idx, apply_sets):
    # fit_idx: rows whose labels may be used; returns features for fit rows (cross-fitted) and for each apply set (direct lookup).
    out_fit = np.zeros((len(fit_idx), len(WINDOWS)), np.float32); out_apply = [np.zeros((len(a), len(WINDOWS)), np.float32) for a in apply_sets]
    inner = list(KFold(5, shuffle=True, random_state=42).split(fit_idx))
    for j, (col, w) in enumerate(WINDOWS):
        v_all = np.r_[X[col].values, X_test[col].values].round().astype(int); vmax = v_all.max()
        v_tr = v_all[:len(X)]
        for a, b in inner:                                       # training rows: statistics from the other 4 inner parts only
            out_fit[b, j] = window_rate(v_tr[fit_idx[a]], y[fit_idx[a]], v_tr[fit_idx[b]], w, vmax)
        for o, rows in zip(out_apply, apply_sets):              # validation / test rows: statistics from the whole training part
            o[:, j] = window_rate(v_tr[fit_idx], y[fit_idx], v_all[rows], w, vmax)
    return out_fit, out_apply

t0 = time.time()
W_tr, (W_va, W_te) = window_features(tr0, [va0, len(X) + np.arange(len(X_test))])
names = [f'win_{c[:6]}_{w}' for c, w in WINDOWS]
print(f'built in {time.time() - t0:.1f}s')
# Leak check: a feature alone should rank training rows (cross-fitted) about as well as validation rows.
print(pd.DataFrame({'feature': names, 'AUC train rows (cross-fitted)': [roc_auc_score(y[tr0], W_tr[:, j]) for j in range(len(names))],
                    'AUC validation rows': [roc_auc_score(y[va0], W_va[:, j]) for j in range(len(names))]}).round(4).to_string(index=False))

built in 0.4s


       feature  AUC train rows (cross-fitted)  AUC validation rows
 win_flight_10                         0.7150               0.7160
 win_flight_50                         0.7067               0.7078
win_flight_200                         0.7052               0.7067
     win_age_2                         0.6803               0.6808
     win_age_5                         0.6753               0.6761
  win_depart_5                         0.5072               0.5083


## 2. Paired XGBoost runs on fold 0

In [3]:
XGB_PARAMS = dict(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                  enable_categorical=True, eval_metric='auc', early_stopping_rounds=50, random_state=42, n_jobs=-1)

def frame(rows_X, te_vals, W=None):
    d = rows_X.copy()
    for c in CAT_COLS: d[c] = d[c].astype('category')
    d['te_flight_distance'] = te_vals
    if W is not None:
        for j, n in enumerate(names): d[n] = W[:, j]
    return d

# In-fold TE of the route, as in 05.
enc = TargetEncoder(cv=5, shuffle=True, random_state=42)
te_tr = enc.fit_transform(X.iloc[tr0][['flight_distance']].astype(str), y[tr0])[:, 0]
te_va = enc.transform(X.iloc[va0][['flight_distance']].astype(str))[:, 0]; te_te = enc.transform(X_test[['flight_distance']].astype(str))[:, 0]
# Early stopping on 10% of the TRAINING part (never on the validation fold).
fit_pos, es_pos = train_test_split(np.arange(len(tr0)), test_size=0.1, stratify=y[tr0], random_state=42)

RES = {}
for tag, use_w in [('ref', False), ('window', True)]:
    d_ck = PREDS_DIR / f'xgb_window{"_ref" if tag == "ref" else ""}_folds'; f = d_ck / 'oof_0.npy'
    if f.exists(): RES[tag] = np.load(f); print(tag, 'loaded'); continue
    Dtr = frame(X.iloc[tr0], te_tr, W_tr if use_w else None); Dva = frame(X.iloc[va0], te_va, W_va if use_w else None)
    Dte = frame(X_test, te_te, W_te if use_w else None)
    t0 = time.time(); m = xgb.XGBClassifier(**XGB_PARAMS)
    m.fit(Dtr.iloc[fit_pos], y[tr0][fit_pos], eval_set=[(Dtr.iloc[es_pos], y[tr0][es_pos])], verbose=0)
    RES[tag] = m.predict_proba(Dva)[:, 1]
    d_ck.mkdir(exist_ok=True); np.save(f, RES[tag]); np.save(d_ck / 'test_0.npy', m.predict_proba(Dte)[:, 1])
    print(f'{tag}: fold 0 AUC {roc_auc_score(y[va0], RES[tag]):.5f} | trees {m.best_iteration} | {(time.time() - t0) / 60:.1f} min', flush=True)
a_ref, a_win = roc_auc_score(y[va0], RES['ref']), roc_auc_score(y[va0], RES['window'])
print(f'\nreference {a_ref:.5f} | + window rates {a_win:.5f} | difference {a_win - a_ref:+.5f}  (acceptance: >= +0.0002)')

ref: fold 0 AUC 0.96070 | trees 243 | 0.2 min


window: fold 0 AUC 0.96068 | trees 279 | 0.2 min



reference 0.96070 | + window rates 0.96068 | difference -0.00002  (acceptance: >= +0.0002)


## 3. Mini-stack on the rows of fold 0
As in `15`–`19`: LR (C = 10) on the logits of the 10 members, inner 5-fold CV over the fold-0 rows; + the window XGBoost.

In [4]:
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
as_logit = lambda o: lg(o) if (o.min() >= 0 and o.max() <= 1) else o
NAMES = {'cat_v1': 'cat_v1', 'cat_v2': 'cat_v2', 'xgb': 'xgb', 'lgbm_tuned': 'lgbm_tuned_final', 'mlp': 'mlp', 'realmlp': 'realmlp_avg',
         'realmlp_rich': 'realmlp_rich_avg', 'tabpfn': 'tabpfn_avg', 'tabpfn_raw': 'tabpfn_raw_avg', 'tabicl': 'tabicl'}
B = np.column_stack([as_logit(np.load(PREDS_DIR / f'oof_{s}.npy').astype(float))[va0] for s in NAMES.values()]); yv = y[va0]
inner = list(StratifiedKFold(5, shuffle=True, random_state=42).split(B, yv))
score = lambda M: np.array([roc_auc_score(yv[b], M[b] @ LogisticRegression(C=10.0, max_iter=500).fit(M[a], yv[a]).coef_[0]) for a, b in inner])
base = score(B)
for tag in RES:
    s_ = score(np.hstack([B, lg(RES[tag])[:, None]])); d_ = s_ - base
    print(f'+ XGB {tag:7s}: mini-stack {s_.mean():.5f} vs {base.mean():.5f} | diff {d_.mean():+.5f} | positive {(d_ > 0).sum()}/5')

+ XGB ref    : mini-stack 0.96192 vs 0.96193 | diff -0.00000 | positive 2/5


+ XGB window : mini-stack 0.96192 vs 0.96193 | diff -0.00001 | positive 3/5


## Results & decisions

**Leak check passed:** every window feature ranks the cross-fitted training rows and the validation rows equally well (e.g. distance ±10: AUC 0.7150 vs 0.7160), so the cross-fitting works.

| Run (fold 0, paired) | AUC | Trees |
|---|---|---|
| XGBoost, `X_final` + route TE (reference) | 0.96070 | 243 |
| **+ 6 window target rates** | **0.96068** (−0.00002) | 279 |

| Mini-stack on fold 0 (10 members = 0.96193) | Difference | Positive inner folds |
|---|---|---|
| + reference XGBoost | −0.00000 | 2/5 |
| + window XGBoost | −0.00001 | 3/5 |

**Decision: rejected.** Windows around a value carry nothing beyond the exact-value target encoding the tree already has — the same verdict as for coarse bins (E15 in `02`). Within one route the label signal is already local; neighbouring routes add no information.

**Side note for the retrospective:** `05` used the validation fold for early stopping of XGBoost (`eval_set=[(X_va, y_va)]`), which makes `oof_xgb` slightly optimistic (typically ~0.00002 AUC; LightGBM in `03`/`04` uses the same pattern). Here the stopping point is chosen on a 10% hold-out of the training part instead; the effect on decisions so far is negligible, but new tree models should follow this pattern.


### Addendum: whole-row "signatures" as keys (same paired XGBoost setup, fold 0; run as a one-off check, code not kept in this notebook)
Hypothesis: the generator reproduces whole *combinations* of values, so rows with the same combination share their fate — something neither single-column TE nor trees (13 splits in a row) see directly. Three features per key: in-fold TE (cross-fitted), log count over train+test, and the satisfied rate of the same key in the **original** survey (label-free).

| Key | Distinct values | Train rows with a twin | Train rows found in original | TE alone (fit / validation AUC) | XGBoost fold 0 vs 0.96070 |
|---|---|---|---|---|---|
| 13-rating signature | 441k | 0.67 | 0.41 | 0.851 / 0.862 | +0.00004 |
| trip: 8 non-rating columns | 416k | 0.73 | 0.18 | 0.757 / 0.771 | +0.00005 |
| trip without delays (6 columns) | 306k | 0.85 | 0.37 | 0.783 / 0.797 | −0.00004 |

Strong on their own, no leak (validation ≥ fit AUC), but **nothing on top of the model**: the tree rebuilds the same information from the individual columns. Also no gain on top of the TabPFN average in a fold-0 blend. Closed.


## Glossary
* **Window target rate** — smoothed satisfied share among rows whose value is within ±w of this row's value.
* **Cross-fitting** — features of training rows computed from statistics that exclude those rows.
* **Early stopping on a training hold-out** — the number of trees is chosen on 10% of the fold's training part, so the validation fold stays untouched.